# 17 — Lite Proper Stacking (4-Model Version)

This notebook is a **lighter proper stacking version** designed to run on a laptop more easily than the larger model-zoo notebooks.

## Purpose
- Keep the professor-facing methodology strong
- Use **rolling time-series CV**
- Use a **true stacked ensemble** based on **out-of-fold probabilities**
- Stay practical enough for a MacBook Air / normal GitHub project folder

## Base learners
- **CatBoost**
- **LightGBM**
- **Logistic Regression**
- **ExtraTrees**

## Meta-learner
- **Multinomial Logistic Regression**

## Expected project structure
Run this notebook from your `notebooks/` folder so these relative paths work:

```text
project-root/
├── data/
│   └── processed/
│       └── supervised_hood_3h_multiclass.csv
├── models/
└── notebooks/
```

## Main saved outputs
- `../models/lite4_stacking_cv_folds.csv`
- `../models/lite4_stacking_cv_summary.csv`
- `../models/lite4_stacking_final_results.csv`

This version intentionally skips:
- feature selection loops
- PCA benchmarking
- permutation importance
- threshold tuning as the main result

That keeps runtime much lower while still defending the ensemble and CV choices.

In [ ]:
from pathlib import Path
import json
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, recall_score, precision_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
import joblib

try:
    from catboost import CatBoostClassifier
except Exception as e:
    raise RuntimeError(
        "CatBoost not available. Install it first, e.g.\n"
        "pip install catboost\n"
        f"Original error: {e}"
    )

try:
    from lightgbm import LGBMClassifier
except Exception as e:
    raise RuntimeError(
        "LightGBM not available. Install it first, e.g.\n"
        "pip install lightgbm\n"
        f"Original error: {e}"
    )

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SUPERVISED_PATH = DATA_DIR / "supervised_hood_3h_multiclass.csv"

OUT_CV_FOLDS = MODEL_DIR / "lite4_stacking_cv_folds.csv"
OUT_CV_SUMMARY = MODEL_DIR / "lite4_stacking_cv_summary.csv"
OUT_FINAL_RESULTS = MODEL_DIR / "lite4_stacking_final_results.csv"
OUT_META_MODEL = MODEL_DIR / "lite4_stacking_meta_model.joblib"
OUT_META_IMPORTANCE = MODEL_DIR / "lite4_stacking_meta_importance.csv"
OUT_CONFIG = MODEL_DIR / "lite4_stacking_config.json"

OUT_CB_MODEL = MODEL_DIR / "lite4_base_cb.cbm"
OUT_LGBM_MODEL = MODEL_DIR / "lite4_base_lgbm.joblib"
OUT_LOGREG_MODEL = MODEL_DIR / "lite4_base_logreg.joblib"
OUT_ET_MODEL = MODEL_DIR / "lite4_base_extratrees.joblib"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TEST_START = pd.Timestamp("2025-07-01 00:00:00")
N_TIME_CHUNKS = 4       # gives 3 rolling validation folds
MIN_TRAIN_CHUNKS = 1

THREADS = min(6, os.cpu_count() or 4)

# Lightweight settings chosen to finish on a MacBook Air more easily.
CB_ITER = 220
LGBM_EST = 220
LOGREG_MAX_ITER = 1200
ET_EST = 120

print("SUPERVISED_PATH:", SUPERVISED_PATH)
print("MODEL_DIR:", MODEL_DIR)
print("THREADS:", THREADS)

In [ ]:
# ------------------------------------------------------------
# Load supervised dataset
# ------------------------------------------------------------
df = pd.read_csv(SUPERVISED_PATH, low_memory=False)

if "time_3h" not in df.columns:
    raise ValueError("Expected a 'time_3h' column in supervised_hood_3h_multiclass.csv")
if "y_class" not in df.columns:
    raise ValueError("Expected a 'y_class' column in supervised_hood_3h_multiclass.csv")

df["time_3h"] = pd.to_datetime(df["time_3h"], errors="coerce")
df = df.loc[df["time_3h"].notna()].copy()

if "HOOD_158_CODE" in df.columns:
    df["HOOD_158_CODE"] = df["HOOD_158_CODE"].astype(str).str.zfill(3)

df["y_class"] = pd.to_numeric(df["y_class"], errors="coerce")
df = df.loc[df["y_class"].notna()].copy()
df["y_class"] = df["y_class"].astype("int8")

sort_cols = ["time_3h"] + (["HOOD_158_CODE"] if "HOOD_158_CODE" in df.columns else [])
df = df.sort_values(sort_cols).reset_index(drop=True)

print("Shape:", df.shape)
print("\nDate range:", df["time_3h"].min(), "to", df["time_3h"].max())
print("\nClass distribution:")
print(df["y_class"].value_counts(normalize=True).sort_index().round(4))

display(df.head())

In [ ]:
# ------------------------------------------------------------
# Development / test split
# Dev  = all rows before 2025-07-01
# Test = rows on or after 2025-07-01
# ------------------------------------------------------------
dev_mask = df["time_3h"] < TEST_START
test_mask = ~dev_mask

y = df["y_class"].copy()
drop_targets = [c for c in ["y_class", "y_count_next"] if c in df.columns]
X = df.drop(columns=drop_targets, errors="ignore").copy()

time_all = df["time_3h"].copy()
X = X.drop(columns=["time_3h"], errors="ignore")

X_dev = X.loc[dev_mask].reset_index(drop=True).copy()
y_dev = y.loc[dev_mask].reset_index(drop=True).copy()
time_dev = time_all.loc[dev_mask].reset_index(drop=True).copy()

X_test = X.loc[test_mask].reset_index(drop=True).copy()
y_test = y.loc[test_mask].reset_index(drop=True).copy()
time_test = time_all.loc[test_mask].reset_index(drop=True).copy()

print("Dev shape:", X_dev.shape, "| Test shape:", X_test.shape)
print("Dev range:", time_dev.min(), "to", time_dev.max())
print("Test range:", time_test.min(), "to", time_test.max())
print("\nDev class distribution:")
print(y_dev.value_counts(normalize=True).sort_index().round(4))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).sort_index().round(4))

In [ ]:
# ------------------------------------------------------------
# Feature typing + helper functions
# ------------------------------------------------------------
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

cat_cols = [c for c in X_dev.columns if X_dev[c].dtype == "object"]
if "HOOD_158_CODE" in X_dev.columns and "HOOD_158_CODE" not in cat_cols:
    cat_cols.append("HOOD_158_CODE")

num_cols = [c for c in X_dev.columns if c not in cat_cols]

for frame in [X_dev, X_test]:
    for c in cat_cols:
        frame[c] = frame[c].astype(str).fillna("missing")

print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))
print("Total features used:", X_dev.shape[1])

def build_preprocessor(num_cols, cat_cols):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_ohe()),
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ], remainder="drop")

def build_logreg_pipe():
    pre = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_ohe()),
        ]), cat_cols),
    ], remainder="drop")
    return Pipeline([
        ("preprocess", pre),
        ("clf", LogisticRegression(
            max_iter=LOGREG_MAX_ITER,
            solver="lbfgs",
            multi_class="multinomial",
            class_weight="balanced",
            random_state=RANDOM_SEED
        ))
    ])

def build_lgbm_pipe():
    pre = build_preprocessor(num_cols, cat_cols)
    return Pipeline([
        ("preprocess", pre),
        ("clf", LGBMClassifier(
            objective="multiclass",
            num_class=3,
            n_estimators=LGBM_EST,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=THREADS,
            verbosity=-1
        ))
    ])

def build_cb_model(y_ref):
    class_counts = y_ref.value_counts().sort_index()
    total = class_counts.sum()
    class_weights = [float(total / (3 * class_counts.get(i, 1))) for i in [0, 1, 2]]
    return CatBoostClassifier(
        loss_function="MultiClass",
        iterations=CB_ITER,
        depth=6,
        learning_rate=0.05,
        eval_metric="TotalF1",
        random_seed=RANDOM_SEED,
        verbose=False,
        thread_count=THREADS,
        class_weights=class_weights
    )

def build_extratrees_pipe():
    pre = build_preprocessor(num_cols, cat_cols)
    return Pipeline([
        ("preprocess", pre),
        ("clf", ExtraTreesClassifier(
            n_estimators=ET_EST,
            max_depth=16,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=THREADS
        ))
    ])


def plot_confusion(cm, labels=(0, 1, 2), title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    plt.show()

def eval_multiclass(y_true, y_pred, proba=None, name="Model", split="test"):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    rec = recall_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    macro_rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    rec_high = recall_score((np.asarray(y_true) == 2).astype(int), (np.asarray(y_pred) == 2).astype(int), zero_division=0)
    prec_high = precision_score((np.asarray(y_true) == 2).astype(int), (np.asarray(y_pred) == 2).astype(int), zero_division=0)
    pred2_rate = float((np.asarray(y_pred) == 2).mean())
    ap2 = np.nan
    if proba is not None:
        ap2 = average_precision_score((np.asarray(y_true) == 2).astype(int), np.asarray(proba)[:, 2])

    return {
        "split": split,
        "model": name,
        "macro_recall": float(macro_rec),
        "recall_0": float(rec[0]),
        "recall_1": float(rec[1]),
        "recall_2": float(rec[2]),
        "precision_2": float(prec_high),
        "pred_class2_rate": float(pred2_rate),
        "ap_class2": float(ap2) if pd.notna(ap2) else np.nan,
        "cm_00": int(cm[0, 0]), "cm_01": int(cm[0, 1]), "cm_02": int(cm[0, 2]),
        "cm_10": int(cm[1, 0]), "cm_11": int(cm[1, 1]), "cm_12": int(cm[1, 2]),
        "cm_20": int(cm[2, 0]), "cm_21": int(cm[2, 1]), "cm_22": int(cm[2, 2]),
    }

def make_time_series_folds(time_series, n_chunks=N_TIME_CHUNKS, min_train_chunks=MIN_TRAIN_CHUNKS):
    uniq_times = pd.Series(pd.to_datetime(time_series)).sort_values().drop_duplicates().to_numpy()
    time_chunks = np.array_split(uniq_times, n_chunks)
    folds = []

    for val_chunk_idx in range(min_train_chunks, len(time_chunks)):
        train_times = np.concatenate(time_chunks[:val_chunk_idx])
        val_times = time_chunks[val_chunk_idx]

        train_idx = np.where(pd.Series(time_series).isin(train_times))[0]
        val_idx = np.where(pd.Series(time_series).isin(val_times))[0]

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        folds.append((train_idx, val_idx))
    return folds

folds = make_time_series_folds(time_dev)
print("Number of usable rolling folds:", len(folds))
for i, (tr_idx, va_idx) in enumerate(folds, start=1):
    print(
        f"Fold {i} | train rows={len(tr_idx):,} | val rows={len(va_idx):,} | "
        f"train max time={time_dev.iloc[tr_idx].max()} | val range={time_dev.iloc[va_idx].min()} to {time_dev.iloc[va_idx].max()}"
    )

In [ ]:
# ------------------------------------------------------------
# Rolling OOF training for base learners
# ------------------------------------------------------------
base_model_builders = {
    "CatBoost": build_cb_model,
    "LightGBM": build_lgbm_pipe,
    "LogReg": build_logreg_pipe,
    "ExtraTrees": build_extratrees_pipe,
}

n_dev = len(X_dev)
n_classes = 3

oof_proba = {
    name: np.full((n_dev, n_classes), np.nan, dtype=float)
    for name in base_model_builders
}
coverage = np.zeros(n_dev, dtype=bool)
fold_rows = []

for fold_num, (tr_idx, va_idx) in enumerate(folds, start=1):
    X_tr = X_dev.iloc[tr_idx].copy()
    y_tr = y_dev.iloc[tr_idx].copy()
    X_va = X_dev.iloc[va_idx].copy()
    y_va = y_dev.iloc[va_idx].copy()

    coverage[va_idx] = True

    print(f"\n===== Fold {fold_num} / {len(folds)} =====")
    print("Train shape:", X_tr.shape, "| Val shape:", X_va.shape)

    for model_name, builder in base_model_builders.items():
        print(f"Training {model_name} on fold {fold_num}...")

        if model_name == "CatBoost":
            model = builder(y_tr)
            model.fit(X_tr, y_tr, cat_features=cat_cols)
            proba_va = model.predict_proba(X_va)
        else:
            model = builder()
            model.fit(X_tr, y_tr)
            proba_va = model.predict_proba(X_va)

        pred_va = np.asarray(proba_va).argmax(axis=1)
        oof_proba[model_name][va_idx] = proba_va

        row = eval_multiclass(y_va, pred_va, proba_va, name=model_name, split=f"fold_{fold_num}")
        row["fold"] = fold_num
        fold_rows.append(row)

fold_df = pd.DataFrame(fold_rows)
display(fold_df)

cv_summary = (
    fold_df.groupby("model", as_index=False)[
        ["macro_recall", "recall_0", "recall_1", "recall_2", "precision_2", "pred_class2_rate", "ap_class2"]
    ]
    .mean()
    .sort_values(["macro_recall", "recall_2"], ascending=False)
    .reset_index(drop=True)
)
display(cv_summary)

covered_idx = np.where(coverage)[0]
print("OOF covered rows:", len(covered_idx), "out of", n_dev)

meta_parts = []
meta_feature_names = []
for model_name in base_model_builders:
    part = oof_proba[model_name][covered_idx]
    if np.isnan(part).any():
        raise ValueError(f"NaNs found in covered OOF block for {model_name}")
    meta_parts.append(part)
    meta_feature_names.extend([f"{model_name}_p0", f"{model_name}_p1", f"{model_name}_p2"])

X_meta_train = np.hstack(meta_parts)
y_meta_train = y_dev.iloc[covered_idx].to_numpy()

print("Meta train shape:", X_meta_train.shape)

In [ ]:
# ------------------------------------------------------------
# Train meta-model on OOF predictions, then refit bases on full dev
# ------------------------------------------------------------
meta_model = LogisticRegression(
    max_iter=1500,
    solver="lbfgs",
    multi_class="multinomial",
    class_weight="balanced",
    random_state=RANDOM_SEED
)
meta_model.fit(X_meta_train, y_meta_train)

final_models = {}
test_meta_parts = []
final_rows = []

for model_name, builder in base_model_builders.items():
    print(f"Refitting {model_name} on full development data...")

    if model_name == "CatBoost":
        model = builder(y_dev)
        model.fit(X_dev, y_dev, cat_features=cat_cols)
        proba_test = model.predict_proba(X_test)
    else:
        model = builder()
        model.fit(X_dev, y_dev)
        proba_test = model.predict_proba(X_test)

    final_models[model_name] = model
    pred_test = np.asarray(proba_test).argmax(axis=1)
    final_rows.append(eval_multiclass(y_test, pred_test, proba_test, name=model_name, split="test"))
    test_meta_parts.append(np.asarray(proba_test))

X_meta_test = np.hstack(test_meta_parts)
proba_stack_test = meta_model.predict_proba(X_meta_test)
pred_stack_test = np.asarray(proba_stack_test).argmax(axis=1)

final_rows.append(eval_multiclass(y_test, pred_stack_test, proba_stack_test, name="Stacking_Meta", split="test"))

final_results = pd.DataFrame(final_rows).sort_values(["macro_recall", "recall_2"], ascending=False).reset_index(drop=True)
display(final_results)

cm_stack = confusion_matrix(y_test, pred_stack_test, labels=[0, 1, 2])
plot_confusion(cm_stack, title="Stacking Meta — Test Confusion Matrix")

coef_abs = np.abs(meta_model.coef_).mean(axis=0)
meta_importance = (
    pd.DataFrame({"meta_feature": meta_feature_names, "mean_abs_coef": coef_abs})
    .assign(model=lambda d: d["meta_feature"].str.replace(r"_p[0-2]$", "", regex=True))
    .groupby("model", as_index=False)["mean_abs_coef"]
    .sum()
    .sort_values("mean_abs_coef", ascending=False)
    .reset_index(drop=True)
)
display(meta_importance)

fold_df.to_csv(OUT_CV_FOLDS, index=False)
cv_summary.to_csv(OUT_CV_SUMMARY, index=False)
final_results.to_csv(OUT_FINAL_RESULTS, index=False)
meta_importance.to_csv(OUT_META_IMPORTANCE, index=False)

joblib.dump(meta_model, OUT_META_MODEL)
final_models["CatBoost"].save_model(str(OUT_CB_MODEL))
joblib.dump(final_models["LightGBM"], OUT_LGBM_MODEL)
joblib.dump(final_models["LogReg"], OUT_LOGREG_MODEL)
joblib.dump(final_models["ExtraTrees"], OUT_ET_MODEL)

config = {
    "random_seed": RANDOM_SEED,
    "test_start": str(TEST_START),
    "n_time_chunks": N_TIME_CHUNKS,
    "threads": THREADS,
    "cb_iter": CB_ITER,
    "lgbm_estimators": LGBM_EST,
    "logreg_max_iter": LOGREG_MAX_ITER,
    "base_models": list(base_model_builders.keys()),
    "paths": {
        "supervised_path": str(SUPERVISED_PATH),
        "cv_folds": str(OUT_CV_FOLDS),
        "cv_summary": str(OUT_CV_SUMMARY),
        "final_results": str(OUT_FINAL_RESULTS),
        "meta_model": str(OUT_META_MODEL),
        "meta_importance": str(OUT_META_IMPORTANCE),
    },
}
config['et_estimators'] = ET_EST

with open(OUT_CONFIG, "w") as f:
    json.dump(config, f, indent=2)

print("\nSaved:")
print("-", OUT_CV_FOLDS)
print("-", OUT_CV_SUMMARY)
print("-", OUT_FINAL_RESULTS)
print("-", OUT_META_MODEL)
print("-", OUT_META_IMPORTANCE)
print("-", OUT_CONFIG)
print("-", OUT_CB_MODEL)
print("-", OUT_LGBM_MODEL)
print("-", OUT_LOGREG_MODEL)
print('-', OUT_ET_MODEL)